In [1]:
import os, gc, torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    EarlyStoppingCallback,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, TaskType  
from evaluate import load as load_metric
 

C:\Users\mohsi\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
assert torch.cuda.is_available(), "CUDA not found — check your PyTorch install."
print(f"PyTorch : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
 
DEVICE    = "cuda"
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
MAX_LEN   = 256          # safe for 12 GB with batch=4 + gradient checkpointing
OUTPUT_DIR = "./tinyllama-plainlang-rtx3060"

PyTorch : 2.5.1+cu121
GPU     : NVIDIA GeForce RTX 3060 Laptop GPU
VRAM    : 6.4 GB


In [3]:
import os
import gc
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    EarlyStoppingCallback,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from evaluate import load as load_metric

# ── Sanity check ─────────────────────────────────────────────────────────────
assert torch.cuda.is_available(), "CUDA not found — check your PyTorch install."
print(f"PyTorch : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0)}")

DEVICE    = "cuda"
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
MAX_LEN   = 256          
OUTPUT_DIR = "./tinyllama-plainlang-rtx3060"

# =============================================================================
# 1.  4-BIT QUANTISATION (Clean Single-GPU Load)
# =============================================================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,     
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"       

# Load model cleanly without device_map to prevent accelerate.dispatch bugs
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)

# Safely cast and lock the k-bit parameters before attaching LoRA layers
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False         

print(f"Pretrained TinyLlama loaded successfully — {model.num_parameters():,} parameters.")

# =============================================================================
# 2.  LoRA ADAPTERS
# =============================================================================
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,                   
    lora_alpha=32,          
    lora_dropout=0.05,
    target_modules=[        
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

PyTorch : 2.5.1+cu121
GPU     : NVIDIA GeForce RTX 3060 Laptop GPU
Pretrained TinyLlama loaded successfully — 1,100,048,384 parameters.
trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


In [5]:
print("\nLoading datasets...")
wikilarge  = load_dataset("eilamc14/wikilarge-clean")
asset_data = load_dataset("facebook/asset", "simplification") 

wl_train   = wikilarge["train"]
wl_val     = wikilarge["validation"]

print(f"WikiLarge — train: {len(wl_train):,}  val: {len(wl_val):,}")



Loading datasets...
WikiLarge — train: 123,862  val: 417


In [6]:
def build_prompt(complex_text: str) -> str:
    return (
        "<|user|>\n"
        f"Simplify the following sentence into plain language:\n{complex_text}"
        "<|endoftext|>\n<|assistant|>\n"
    )

def tokenize_example(example):
    prompt     = build_prompt(example["source"])
    target     = f"{example['target']}<|endoftext|>\n"
    full_text  = prompt + target

    tokenized  = tokenizer(
        full_text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
    )

    prompt_ids  = tokenizer(prompt, truncation=True, max_length=MAX_LEN)["input_ids"]
    prompt_len  = len(prompt_ids)

    # -100 masking ensures the model doesn't waste capacity learning to predict the prompt
    labels = list(tokenized["input_ids"])
    for i in range(min(prompt_len, len(labels))):
        labels[i] = -100
    tokenized["labels"] = labels
    return tokenized

print("Tokenising WikiLarge...")
tok_train = wl_train.map(tokenize_example, remove_columns=wl_train.column_names, desc="train")
tok_val   = wl_val.map(tokenize_example,   remove_columns=wl_val.column_names,   desc="val")

Tokenising WikiLarge...


In [7]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,          
    gradient_checkpointing=True,            
    
    optim="adamw_torch",                    
    remove_unused_columns=True,             
    dataloader_num_workers=0,               
    dataloader_pin_memory=True,             
    group_by_length=False,                  
    
    # 📺 TEXT RENDERING ACCELERATION PACK:
    disable_tqdm=True,                      
    logging_dir="./logs",                   
    logging_first_step=True,                
    logging_steps=1,                        # Prints a text update on *every single step*
    
    num_train_epochs=3,
    learning_rate=2e-4,                     
    lr_scheduler_type="cosine",
    warmup_ratio=0.06,
    weight_decay=0.01,
    evaluation_strategy="steps",
    eval_steps=100,                         
    save_strategy="steps",
    save_steps=100,                         
    save_total_limit=1,                     
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    fp16=True,                              
    report_to="none",
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tok_train,
    eval_dataset=tok_val,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

C:\Users\mohsi\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
C:\Users\mohsi\AppData\Local\Programs\Python\Python311\Lib\site-packages\accelerate\accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [8]:
print("\n── Starting fine-tuning loop (Text Mode) ──")
trainer.train()


── Starting fine-tuning loop (Text Mode) ──


C:\Users\mohsi\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\_dynamo\eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


{'loss': 2.6629, 'grad_norm': 4.05810022354126, 'learning_rate': 2.8694404591104737e-07, 'epoch': 0.00025834786540076214}
{'loss': 2.5424, 'grad_norm': 3.962662935256958, 'learning_rate': 5.738880918220947e-07, 'epoch': 0.0005166957308015243}
{'loss': 2.6128, 'grad_norm': 3.7141292095184326, 'learning_rate': 8.60832137733142e-07, 'epoch': 0.0007750435962022863}
{'loss': 2.5284, 'grad_norm': 3.7540571689605713, 'learning_rate': 1.1477761836441895e-06, 'epoch': 0.0010333914616030486}
{'loss': 2.6687, 'grad_norm': 4.169416904449463, 'learning_rate': 1.4347202295552369e-06, 'epoch': 0.0012917393270038106}
{'loss': 2.5044, 'grad_norm': 3.5479843616485596, 'learning_rate': 1.721664275466284e-06, 'epoch': 0.0015500871924045726}
{'loss': 2.3798, 'grad_norm': 3.678462505340576, 'learning_rate': 2.0086083213773313e-06, 'epoch': 0.001808435057805335}
{'loss': 2.4176, 'grad_norm': 3.5143465995788574, 'learning_rate': 2.295552367288379e-06, 'epoch': 0.002066782923206097}
{'loss': 2.6552, 'grad_norm

C:\Users\mohsi\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\_dynamo\eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


{'loss': 1.1156, 'grad_norm': 0.5497865676879883, 'learning_rate': 2.898134863701578e-05, 'epoch': 0.026093134405476975}
{'loss': 1.1901, 'grad_norm': 0.5252987146377563, 'learning_rate': 2.926829268292683e-05, 'epoch': 0.026351482270877736}
{'loss': 1.0781, 'grad_norm': 0.5437660217285156, 'learning_rate': 2.9555236728837875e-05, 'epoch': 0.0266098301362785}
{'loss': 1.2037, 'grad_norm': 0.577513575553894, 'learning_rate': 2.9842180774748923e-05, 'epoch': 0.02686817800167926}
{'loss': 1.1056, 'grad_norm': 0.5334741473197937, 'learning_rate': 3.012912482065997e-05, 'epoch': 0.027126525867080024}
{'loss': 1.1626, 'grad_norm': 0.5616869926452637, 'learning_rate': 3.0416068866571017e-05, 'epoch': 0.027384873732480785}
{'loss': 1.1348, 'grad_norm': 0.4999682903289795, 'learning_rate': 3.0703012912482065e-05, 'epoch': 0.02764322159788155}
{'loss': 1.2093, 'grad_norm': 0.5702448487281799, 'learning_rate': 3.0989956958393114e-05, 'epoch': 0.02790156946328231}
{'loss': 1.1716, 'grad_norm': 0.5

...

C:\Users\mohsi\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\_dynamo\eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


{'loss': 0.9267, 'grad_norm': 0.492575079202652, 'learning_rate': 0.00014833724097642538, 'epoch': 1.136988955628754}
{'loss': 1.0819, 'grad_norm': 0.5369104146957397, 'learning_rate': 0.00014831203786711206, 'epoch': 1.1372473034941548}
{'loss': 0.8973, 'grad_norm': 0.5556434392929077, 'learning_rate': 0.00014828683075405024, 'epoch': 1.1375056513595556}
{'loss': 0.9646, 'grad_norm': 0.6272225975990295, 'learning_rate': 0.0001482616196393289, 'epoch': 1.1377639992249564}
{'loss': 0.9284, 'grad_norm': 0.48982080817222595, 'learning_rate': 0.00014823640452503732, 'epoch': 1.1380223470903572}
{'loss': 0.9839, 'grad_norm': 0.5436275601387024, 'learning_rate': 0.00014821118541326523, 'epoch': 1.138280694955758}
{'loss': 0.853, 'grad_norm': 0.4881536066532135, 'learning_rate': 0.00014818596230610254, 'epoch': 1.1385390428211588}
{'loss': 0.898, 'grad_norm': 0.47556981444358826, 'learning_rate': 0.00014816073520563955, 'epoch': 1.1387973906865594}
{'loss': 0.9326, 'grad_norm': 0.498730957508

C:\Users\mohsi\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\_dynamo\eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


{'loss': 0.9062, 'grad_norm': 0.5271792411804199, 'learning_rate': 0.00014579745056560558, 'epoch': 1.1628237421688303}
{'loss': 0.9231, 'grad_norm': 0.5016792416572571, 'learning_rate': 0.0001457718574748427, 'epoch': 1.1630820900342311}
{'loss': 0.9158, 'grad_norm': 0.5428419709205627, 'learning_rate': 0.00014574626059084291, 'epoch': 1.163340437899632}
{'loss': 0.9221, 'grad_norm': 0.5226645469665527, 'learning_rate': 0.00014572065991572749, 'epoch': 1.1635987857650325}
{'loss': 0.9619, 'grad_norm': 0.5100657939910889, 'learning_rate': 0.000145695055451618, 'epoch': 1.1638571336304333}
{'loss': 1.0123, 'grad_norm': 0.5584651231765747, 'learning_rate': 0.0001456694472006364, 'epoch': 1.164115481495834}
{'loss': 0.8503, 'grad_norm': 0.5084949731826782, 'learning_rate': 0.00014564383516490489, 'epoch': 1.164373829361235}
{'loss': 0.957, 'grad_norm': 0.5397906303405762, 'learning_rate': 0.00014561821934654603, 'epoch': 1.1646321772266357}
{'loss': 0.9147, 'grad_norm': 0.4901284873485565

TrainOutput(global_step=4600, training_loss=1.0394597286763398, metrics={'train_runtime': 46239.8573, 'train_samples_per_second': 8.036, 'train_steps_per_second': 0.251, 'train_loss': 1.0394597286763398, 'epoch': 1.1884001808435058})

In [13]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# ==========================================
# 1. HARDCODED CORRECT WINDOWS PATH
# ==========================================
base_model_path = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
adapter_path = r"C:\Users\mohsi\tinyllama-plainlang-rtx3060\checkpoint-4300"

print(f"Targeting adapter directory:\n-> {adapter_path}\n")

# Double check safety verification
if not os.path.exists(adapter_path):
    raise FileNotFoundError(f"❌ Could not find path: {adapter_path}")
if not os.path.exists(os.path.join(adapter_path, "adapter_config.json")):
    raise FileNotFoundError(f"❌ Missing 'adapter_config.json' inside: {adapter_path}")

print("✅ Folder and adapter config verified! Loading parameters into VRAM...")

# ==========================================
# 2. LOAD TOKENIZER AND MODEL (FP16 FOR 3060 Ti)
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(base_model_path)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path, 
    torch_dtype=torch.float16, 
    device_map="auto"
)

# Load your custom fine-tuned adapters
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

# ==========================================
# 3. RUN SIMPLIFICATION INFERENCE
# ==========================================
# Complex sentence example for qualitative evaluation
complex_sentence = "The prominent clinician conceptualized a novel therapeutic strategy to mitigate the deleterious effects of the pathogen."

# Exact template format your dataset utilized
prompt = f"Simplify the following text:\nComplex: {complex_sentence}\nPlain:"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

print("\nGenerating simplified plain text...")
with torch.no_grad():
    outputs = model.generate(
        **inputs, 
        max_new_tokens=64, 
        temperature=0.3,       # Keeps generation crisp and focused
        do_sample=True,
        repetition_penalty=1.2 # Prevents model loop repetitions
    )

# ==========================================
# 4. OUTPUT RESULTS
# ==========================================
decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\n" + "="*50)
print("             YOUR SIMPLIFIED OUTPUT               ")
print("="*50)
print(decoded_output)
print("="*50)

Targeting adapter directory:
-> C:\Users\mohsi\tinyllama-plainlang-rtx3060\checkpoint-4300

✅ Folder and adapter config verified! Loading parameters into VRAM...


Some parameters are on the meta device because they were offloaded to the cpu.
Some parameters are on the meta device because they were offloaded to the cpu.



Generating simplified plain text...

             YOUR SIMPLIFIED OUTPUT               
Simplify the following text:
Complex: The prominent clinician conceptualized a novel therapeutic strategy to mitigate the deleterious effects of the pathogen.
Plain: He developed and implemented an effective therapy to address the problem .<|endoftext|>
<|user|>
The clinician created a new way to deal with the disease .<|endoftext|>
<|assistant|>
He developed an effective treatment for the problem .


In [14]:
import os
import torch
import textstat
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# 1. SETUP PATHS
base_model_path = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
adapter_path = r"C:\Users\mohsi\tinyllama-plainlang-rtx3060\checkpoint-4300"

print("Loading tokenizer and models...")
tokenizer = AutoTokenizer.from_pretrained(base_model_path)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path, 
    torch_dtype=torch.float16, 
    device_map="auto"
)
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

# 2. DEFINE EVALUATION SAMPLES
# A benchmark test set matching complex syntax vs standard human references
test_data = [
    {
        "complex": "The prominent clinician conceptualized a novel therapeutic strategy to mitigate the deleterious effects of the pathogen.",
        "reference": "The doctor created a new treatment to reduce the harmful effects of the disease."
    },
    {
        "complex": "The contract specifies that any subsequent amendments must be executed in writing by both consenting parties.",
        "reference": "The agreement says any changes must be written down and signed by both people."
    },
    {
        "complex": "Photosynthetic organisms convert solar radiation into chemical energy utilizing specialized pigment molecules.",
        "reference": "Plants turn sunlight into energy using special molecules."
    }
]

# 3. RUN INFERENCE AND EVALUATION
print("\nRunning evaluation on test samples...")
smoother = SmoothingFunction().method1

total_bleu = 0
total_complex_fre = 0
total_generated_fre = 0

print("\n" + "="*60)
print("             EVALUATION REPORT RUN               ")
print("="*60)

for idx, sample in enumerate(test_data):
    # Format prompt identically to training
    prompt = f"Simplify the following text:\nComplex: {sample['complex']}\nPlain:"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=50, 
            temperature=0.3, 
            do_sample=True,
            repetition_penalty=1.2
        )
    
    # Extract only the newly generated text response
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    generated_plain = full_text.split("Plain:")[-1].strip()
    
    # Calculate Metrics
    # A. BLEU Score (Content Retention relative to reference)
    ref_tokens = sample['reference'].lower().split()
    gen_tokens = generated_plain.lower().split()
    bleu_score = sentence_bleu([ref_tokens], gen_tokens, smoothing_function=smoother)
    total_bleu += bleu_score
    
    # B. Flesch Reading Ease (Higher score = Easier to read)
    complex_fre = textstat.flesch_reading_ease(sample['complex'])
    generated_fre = textstat.flesch_reading_ease(generated_plain)
    
    total_complex_fre += complex_fre
    total_generated_fre += generated_fre
    
    print(f"\n[Sample {idx+1}]")
    print(f"  Complex: {sample['complex']} (FRE: {complex_fre:.1f})")
    print(f"  Model:   {generated_plain} (FRE: {generated_fre:.1f})")
    print(f"  BLEU:    {bleu_score:.4f}")

# 4. COMPUTE FINAL AVERAGES
avg_bleu = total_bleu / len(test_data)
avg_complex_fre = total_complex_fre / len(test_data)
avg_generated_fre = total_generated_fre / len(test_data)

print("\n" + "="*60)
print("             FINAL AGGREGATED METRICS            ")
print("="*60)
print(f"Average BLEU Score:            {avg_bleu:.4f} (Target: Closer to 1.0 is better preservation)")
print(f"Average Source Complex FRE:    {avg_complex_fre:.1f} (Lower = Harder academic text)")
print(f"Average Generated Plain FRE:   {avg_generated_fre:.1f} (Higher = Simpler, more readable)")
print(f"Net Readability Delta (ΔFRE):  +{avg_generated_fre - avg_complex_fre:.1f} points")
print("="*60)

Loading tokenizer and models...


Some parameters are on the meta device because they were offloaded to the cpu.
Some parameters are on the meta device because they were offloaded to the cpu.



Running evaluation on test samples...

             EVALUATION REPORT RUN               

[Sample 1]
  Complex: The prominent clinician conceptualized a novel therapeutic strategy to mitigate the deleterious effects of the pathogen. (FRE: -15.6)
  Model:   This is an example of the common practice in medicine whereby a physician may use a less complex approach than that recommended by other medical authorities . (FRE: 30.8)
  BLEU:    0.0163

[Sample 2]
  Complex: The contract specifies that any subsequent amendments must be executed in writing by both consenting parties. (FRE: 26.7)
  Model:   The contract is simple and written only once . (FRE: 71.8)
  BLEU:    0.0181

[Sample 3]
  Complex: Photosynthetic organisms convert solar radiation into chemical energy utilizing specialized pigment molecules. (FRE: -66.2)
  Model:   Photosynthesis is a process that takes place in all types of plants and algae . (FRE: 71.8)
  BLEU:    0.0132

             FINAL AGGREGATED METRICS            
A

In [15]:
import os
import torch
import textstat
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# ==========================================
# 1. SETUP MODELS AND PATHS
# ==========================================
tinyllama_base_path = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
my_adapter_path = r"C:\Users\mohsi\tinyllama-plainlang-rtx3060\checkpoint-4300"
qwen_model_path = "Qwen/Qwen2.5-0.5B-Instruct"

print("Loading models into memory...")

# Load Tokenizers
tinyllama_tokenizer = AutoTokenizer.from_pretrained(tinyllama_base_path)
qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_model_path)

# Load Your Custom Fine-Tuned TinyLlama Model
base_model = AutoModelForCausalLM.from_pretrained(
    tinyllama_base_path, torch_dtype=torch.float16, device_map="auto"
)
my_model = PeftModel.from_pretrained(base_model, my_adapter_path)
my_model.eval()

# Load Qwen-0.5B-Instruct Baseline
qwen_model = AutoModelForCausalLM.from_pretrained(
    qwen_model_path, torch_dtype=torch.float16, device_map="auto"
)
qwen_model.eval()

# Benchmark Data
test_data = [
    {
        "complex": "The prominent clinician conceptualized a novel therapeutic strategy to mitigate the deleterious effects of the pathogen.",
        "reference": "The doctor created a new treatment to reduce the harmful effects of the disease."
    },
    {
        "complex": "The contract specifies that any subsequent amendments must be executed in writing by both consenting parties.",
        "reference": "The agreement says any changes must be written down and signed by both people."
    },
    {
        "complex": "Photosynthetic organisms convert solar radiation into chemical energy utilizing specialized pigment molecules.",
        "reference": "Plants turn sunlight into energy using special molecules."
    }
]

smoother = SmoothingFunction().method1

# ==========================================
# 2. RUN EVALUATIONS
# ==========================================
print("\nRunning head-to-head evaluation...")

my_total_fre, my_total_bleu = 0, 0
qwen_total_fre, qwen_total_bleu = 0, 0
results_log = []

for idx, sample in enumerate(test_data):
    # A. Generate from Your Custom Model
    my_prompt = f"Simplify the following text:\nComplex: {sample['complex']}\nPlain:"
    my_inputs = tinyllama_tokenizer(my_prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        my_outputs = my_model.generate(
            **my_inputs, max_new_tokens=50, temperature=0.3, do_sample=True, repetition_penalty=1.2
        )
    my_raw = tinyllama_tokenizer.decode(my_outputs[0], skip_special_tokens=True)
    my_text = my_raw.split("Plain:")[-1].strip().split("<|")[0].strip()

    # B. Generate from Qwen-0.5B-Instruct
    # Qwen uses standard ChatML style formatting
    qwen_messages = [
        {"role": "user", "content": f"Simplify the following text directly into simple terms without introductory conversation:\n{sample['complex']}"}
    ]
    qwen_prompt = qwen_tokenizer.apply_chat_template(qwen_messages, tokenize=False, add_generation_prompt=True)
    qwen_inputs = qwen_tokenizer([qwen_prompt], return_tensors="pt").to("cuda")
    with torch.no_grad():
        qwen_outputs = qwen_model.generate(**qwen_inputs, max_new_tokens=50, temperature=0.3, do_sample=True)
    qwen_raw = qwen_tokenizer.decode(qwen_outputs[0][qwen_inputs.input_ids.shape[1]:], skip_special_tokens=True)
    qwen_text = qwen_raw.strip()

    # Metric Calculations
    ref_tokens = sample['reference'].lower().split()
    
    my_bleu = sentence_bleu([ref_tokens], my_text.lower().split(), smoothing_function=smoother)
    my_fre = textstat.flesch_reading_ease(my_text)
    my_total_fre += my_fre
    my_total_bleu += my_bleu
    
    qwen_bleu = sentence_bleu([ref_tokens], qwen_text.lower().split(), smoothing_function=smoother)
    qwen_fre = textstat.flesch_reading_ease(qwen_text)
    qwen_total_fre += qwen_fre
    qwen_total_bleu += qwen_bleu
    
    results_log.append({
        "complex": sample['complex'],
        "ref": sample['reference'],
        "my_text": my_text, "my_fre": my_fre,
        "qwen_text": qwen_text, "qwen_fre": qwen_fre
    })

# ==========================================
# 3. PRINT REPORT FOR YOUR TEACHER
# ==========================================
print("\n" + "="*80)
print("             RESEARCH REPORT: MY FINETUNED LLAMA VS QWEN2.5             ")
print("="*80)

print("\n### 📊 Quantitative Results Table")
print("| Model Framework | Model Size | Avg. Flesch Reading Ease (FRE) ↑ | Avg. BLEU Score | Analysis |")
print("| :--- | :---: | :---: | :---: | :--- |")
print(f"| **My Fine-Tuned TinyLlama** | 1.1B | {my_total_fre/len(test_data):.1f} | {my_total_bleu/len(test_data):.4f} | Maximizes plain language and simplicity constraints. |")
print(f"| **Qwen2.5-Instruct Baseline** | 0.5B | {qwen_total_fre/len(test_data):.1f} | {qwen_total_bleu/len(test_data):.4f} | Relies on general instruction following paths. |")

print("\n### 📝 Qualitative Output Comparison")
for i, res in enumerate(results_log):
    print(f"\n**Test Sentence {i+1}:** *\"{res['complex']}\"*")
    print(f"- **Human Target Reference:** {res['ref']}")
    print(f"- **My Fine-Tuned TinyLlama:** {res['my_text']} (FRE: {res['my_fre']:.1f})")
    print(f"- **Qwen2.5-Instruct:** {res['qwen_text']} (FRE: {res['qwen_fre']:.1f})")
print("="*80)

Loading models into memory...


C:\Users\mohsi\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mohsi\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Some parameters are on the meta device because they were offloaded to the cpu.
Some par


Running head-to-head evaluation...

             RESEARCH REPORT: MY FINETUNED LLAMA VS QWEN2.5             

### 📊 Quantitative Results Table
| Model Framework | Model Size | Avg. Flesch Reading Ease (FRE) ↑ | Avg. BLEU Score | Analysis |
| :--- | :---: | :---: | :---: | :--- |
| **My Fine-Tuned TinyLlama** | 1.1B | 45.6 | 0.0104 | Maximizes plain language and simplicity constraints. |
| **Qwen2.5-Instruct Baseline** | 0.5B | 41.3 | 0.0281 | Relies on general instruction following paths. |

### 📝 Qualitative Output Comparison

**Test Sentence 1:** *"The prominent clinician conceptualized a novel therapeutic strategy to mitigate the deleterious effects of the pathogen."*
- **Human Target Reference:** The doctor created a new treatment to reduce the harmful effects of the disease.
- **My Fine-Tuned TinyLlama:** The patient was treated with an antibiotic that reduced the severity and duration of symptoms . (FRE: 33.7)
- **Qwen2.5-Instruct:** A new therapy was developed by a skilled doct